# Notebook 3: Analysis & Figures

Tests all three pre-registered hypotheses and generates publication-quality figures.

**Read `PREREGISTRATION.md` before running.** All analysis choices here were committed
before data collection began.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from pathlib import Path

df = pd.read_csv('../data/processed/mention_share.csv')

# One row per election for H1 (whether attention leader won)
elections = df.sort_values('popular_vote_winner', ascending=False).drop_duplicates('year').copy()
elections['attention_predicted_winner'] = (elections['attention_leader'] == elections['popular_vote_winner']).astype(int)

print(f'{len(elections)} elections loaded')
print(elections[['year', 'candidate', 'mention_share', 'popular_vote_share', 'attention_leader', 'popular_vote_winner']])

## H1: Binomial Test
Does the attention leader win the popular vote more often than chance?

In [ ]:
# Per pre-registration: one row per election, attention_leader == popular_vote_winner
per_election = (
    df.groupby('year')
    .apply(lambda g: int((g.loc[g['attention_leader']==1, 'popular_vote_winner'] == 1).any()))
    .reset_index(name='attention_won')
)

k = per_election['attention_won'].sum()  # elections where attention leader won
n = len(per_election)

result = stats.binomtest(k, n=n, p=0.5, alternative='two-sided')

print(f'Elections where attention leader won popular vote: {k}/{n} ({k/n:.1%})')
print(f'Binomial test p-value: {result.pvalue:.4f}')
print(f'95% CI: [{result.proportion_ci().low:.3f}, {result.proportion_ci().high:.3f}]')
print()
if result.pvalue < 0.05:
    print('H1: SUPPORTED — attention leader wins significantly more than chance')
else:
    print('H1: NOT SUPPORTED at α=0.05')

In [ ]:
# Figure 1: Win rate bar chart with CI
fig = go.Figure()

fig.add_trace(go.Bar(
    x=['Attention leader', 'Chance baseline'],
    y=[k/n, 0.5],
    error_y=dict(
        type='data',
        array=[result.proportion_ci().high - k/n, 0],
        arrayminus=[k/n - result.proportion_ci().low, 0],
    ),
    marker_color=['#1a1a2e', '#888888'],
    text=[f'{k/n:.0%}', '50%'],
    textposition='outside',
))

fig.update_layout(
    title='H1: How often does the attention leader win the popular vote?',
    yaxis=dict(title='Win rate', range=[0, 1.1], tickformat='.0%'),
    xaxis=dict(title=''),
    plot_bgcolor='white',
    width=500, height=450,
)
fig.show()

## H2: Mention Share vs. Vote Share Correlation

In [ ]:
r, p = stats.pearsonr(df['mention_share'], df['popular_vote_share'])

print(f'Pearson r = {r:.3f}, p = {p:.4f} (n={len(df)})')
if p < 0.05:
    print('H2: SUPPORTED — mention share significantly correlates with vote share')
else:
    print('H2: NOT SUPPORTED at α=0.05')

In [ ]:
# Figure 2: Scatter plot — mention share vs. vote share
df['label'] = df['candidate'] + ' ' + df['year'].astype(str)
df['color'] = df['party'].map({'D': '#0047AB', 'R': '#C8102E'})

# Regression line
m, b = np.polyfit(df['mention_share'], df['popular_vote_share'], 1)
x_range = np.linspace(df['mention_share'].min(), df['mention_share'].max(), 100)

fig = go.Figure()

for party, color, name in [('D', '#0047AB', 'Democrat'), ('R', '#C8102E', 'Republican')]:
    subset = df[df['party'] == party]
    fig.add_trace(go.Scatter(
        x=subset['mention_share'],
        y=subset['popular_vote_share'],
        mode='markers+text',
        text=subset['label'],
        textposition='top center',
        textfont=dict(size=9),
        marker=dict(color=color, size=10),
        name=name,
    ))

fig.add_trace(go.Scatter(
    x=x_range, y=m * x_range + b,
    mode='lines',
    line=dict(color='gray', dash='dash'),
    name=f'OLS fit (r={r:.2f})',
    showlegend=True,
))

fig.update_layout(
    title=f'H2: Mention Share vs. Popular Vote Share (r = {r:.2f}, p = {p:.3f})',
    xaxis=dict(title='Mention share (12 months pre-election)', tickformat='.0%'),
    yaxis=dict(title='Popular vote share', tickformat='.0%'),
    plot_bgcolor='white',
    width=800, height=600,
)
fig.show()

## H3 (Exploratory): Mention Gap vs. Vote Margin

In [ ]:
# Build one-row-per-election dataframe for H3
winners = df[df['popular_vote_winner'] == 1][['year', 'candidate', 'mention_share', 'popular_vote_share']].copy()
losers = df[df['popular_vote_winner'] == 0][['year', 'candidate', 'mention_share', 'popular_vote_share']].copy()

h3 = winners.merge(losers, on='year', suffixes=('_winner', '_loser'))
h3['mention_gap'] = h3['mention_share_winner'] / h3['mention_share_loser']
h3['vote_margin'] = h3['popular_vote_share_winner'] - h3['popular_vote_share_loser']

r3, p3 = stats.pearsonr(h3['mention_gap'], h3['vote_margin'])
print(f'H3 (exploratory): mention gap vs. vote margin: r = {r3:.3f}, p = {p3:.4f}')

fig = px.scatter(
    h3, x='mention_gap', y='vote_margin',
    text='year',
    title=f'H3 (Exploratory): Mention Gap vs. Vote Margin (r = {r3:.2f})',
    labels={'mention_gap': 'Winner mention share / Loser mention share', 'vote_margin': 'Vote margin (winner - loser)'},
    trendline='ols',
)
fig.update_traces(textposition='top center')
fig.update_layout(plot_bgcolor='white', yaxis=dict(tickformat='.1%'))
fig.show()

## Pre-specified Sensitivity Analyses

In [ ]:
# 1. Exclude incumbent elections
df_no_inc = df[df['incumbent_running'] == 0]
per_election_no_inc = (
    df_no_inc.groupby('year')
    .apply(lambda g: int((g.loc[g['attention_leader']==1, 'popular_vote_winner'] == 1).any()))
    .reset_index(name='attention_won')
)
k2 = per_election_no_inc['attention_won'].sum()
n2 = len(per_election_no_inc)
r2 = stats.binomtest(k2, n=n2, p=0.5, alternative='two-sided')
print(f'Excluding incumbents: {k2}/{n2} ({k2/n2:.1%}), p={r2.pvalue:.4f}')

# 2. GDELT-only years (1979+)
df_gdelt = df[df['source'] == 'GDELT']
per_election_gdelt = (
    df_gdelt.groupby('year')
    .apply(lambda g: int((g.loc[g['attention_leader']==1, 'popular_vote_winner'] == 1).any()))
    .reset_index(name='attention_won')
)
k3 = per_election_gdelt['attention_won'].sum()
n3 = len(per_election_gdelt)
r3_binom = stats.binomtest(k3, n=n3, p=0.5, alternative='two-sided')
print(f'GDELT-only (1979+): {k3}/{n3} ({k3/n3:.1%}), p={r3_binom.pvalue:.4f}')

# 3. Exclude 1992 (Perot denominator effect)
df_no92 = df[df['year'] != 1992]
r92, p92 = stats.pearsonr(df_no92['mention_share'], df_no92['popular_vote_share'])
print(f'Excluding 1992 (Perot): H2 r={r92:.3f}, p={p92:.4f}')

## Figure 3: Timeline of Mention Ratios

How big was the attention gap in each election, and did the bigger-attention candidate win?

In [ ]:
h3_sorted = h3.sort_values('year')

fig = go.Figure()

colors = ['#2ecc71' if row['candidate_winner'] == row['candidate_winner'] else '#e74c3c' 
          for _, row in h3_sorted.iterrows()]  # placeholder; update after merge

fig.add_trace(go.Bar(
    x=h3_sorted['year'],
    y=h3_sorted['mention_gap'],
    text=[f"{row['candidate_winner']} {row['mention_gap']:.1f}×" for _, row in h3_sorted.iterrows()],
    textposition='outside',
    marker_color='#1a1a2e',
))

fig.add_hline(y=1, line_dash='dash', line_color='gray', annotation_text='Equal attention')

fig.update_layout(
    title='Attention Gap by Election Year (winner mentions / loser mentions)',
    xaxis=dict(title='Election Year', tickmode='array', tickvals=h3_sorted['year'].tolist()),
    yaxis=dict(title='Mention ratio (winner / loser)'),
    plot_bgcolor='white',
    width=900, height=500,
)
fig.show()